# Configuration

In [ ]:
# Cell: Config (edit values as needed)
import os
from pathlib import Path

MODE = "lora"

BASE_MODEL = "meta-llama/Llama-2-7b-hf"

TOKENIZER_DIR = "./"  
TOKENIZER_FILES = {
    "tokenizer_json": "tokenizer.json",
    "tokenizer_model": "tokenizer.model",
    "tokenizer_config": "tokenizer_config.json",
    "special_tokens_map": "special_tokens_map.json",
    "generation_config": "generation_config.json",
    "model_config": "config.json"  
}


MERGED_JSONL = "data/merged_medical.jsonl"
HF_DATASETS = {
    "epfl_guidelines": "epfl-llm/guidelines",
    "redpajama_arxiv": ("togethercomputer/RedPajama-Data-1T", "arxiv"),
    "s2orc_biomed": ("s2orc", "biomed")
}
KAGGLE_CSV = "data/doctor_healthcare_100k.csv"


OUTPUT_DIR = "F-Chat-Medical-Model" 
CHECKPOINT_DIR = OUTPUT_DIR + "-Checkpoints"
QUANT_DIR = OUTPUT_DIR + "-quantized" 

TRAINING_ARGS = {
    "num_train_epochs": 2,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "fp16": True,
    "logging_steps": 20,
    "save_steps": 1000,
    "save_total_limit": 2,
    "warmup_steps": 200,
    "seed": 42,
}


LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "target_modules": ["q_proj", "v_proj"],
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": "CAUSAL_LM"
}

NUM_PROC = max(1, (os.cpu_count() or 1) // 2)
os.makedirs("data", exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(QUANT_DIR, exist_ok=True)

print(" Stage 0: Configuration loaded.")

# Imports

In [ ]:
# Cell: imports
import json
import re
import math
import random
from pathlib import Path
from typing import Iterator, Dict, Any, List
import csv

import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    Trainer, 
    TrainingArguments, 
    DataCollatorForLanguageModeling, 
    set_seed
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from safetensors.torch import save_file

# Try to import auto-gptq, will check later
try:
    from auto_gptq import AutoGPTQForCausalLM
    HAS_GPTQ = True
except ImportError:
    HAS_GPTQ = False
    print("[WARN] auto-gptq library not found. Quantization (Stage 7/8) will be skipped.")

print(" Stage 1: Imports complete.")

# Data Collection & Merge

In [ ]:
# Cell: Data helper functions
def clean_text(s: str) -> str:
    if s is None:
        return ""
    s = s.replace("\r", " ").replace("\n", " ").strip()
    s = re.sub(r"<[^>]+>", " ", s) # remove html tags
    s = re.sub(r"\s+", " ", s) # remove multiple spaces
    return s

def hf_stream_text(dataset_name: str, split="train", field_candidates=None, fraction: float = None):
    print(f"[DATA] loading {dataset_name} split={split}")
    # Handle tuple dataset_name (name, subset)
    if isinstance(dataset_name, tuple):
        ds = load_dataset(dataset_name[0], dataset_name[1], split=split, streaming=True)
    else:
        ds = load_dataset(dataset_name, split=split, streaming=True)
    
    if fraction:
        print(f"[DATA] Sampling {fraction*100}% of {dataset_name}")
        ds = ds.shuffle(seed=42).take(int(fraction * 100000)) # Approx sampling for streaming

    for item in ds:
        txt = ""
        if field_candidates:
            for f in field_candidates:
                if f in item and item[f]:
                    txt = item[f]
                    break
        else:
            txt = item.get("text") or item.get("article") or item.get("abstract") or ""
        
        txt = clean_text(txt)
        if len(txt) < 20:
            continue
        yield txt

def csv_stream_qas(csv_path: str, q_col="question", a_col="answer"):
    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for r in reader:
            q = clean_text(r.get(q_col,""))
            a = clean_text(r.get(a_col,""))
            if not q or not a:
                continue
            yield f"Q: {q} A: {a}"

# Cell: Build merged JSONL
def build_merged_jsonl(out_path="data/merged_medical.jsonl", dev_mode=True):
    out_path = Path(out_path)
    if out_path.exists():
        print("[DATA] merged file already exists:", out_path)
        return str(out_path)
    
    print("[DATA] Building merged JSONL at", out_path)
    with open(out_path, "w", encoding="utf-8") as fo:
        
        # 1) RedPajama (take small sample in dev)
        frac = 0.001 if dev_mode else None # 0.001 of arxiv is still large
        for txt in hf_stream_text(HF_DATASETS["redpajama_arxiv"], field_candidates=["text","article","content"], fraction=frac):
            fo.write(json.dumps({"text": txt}, ensure_ascii=False) + "\n")
        
        # 2) s2orc biomed
        frac = 0.01 if dev_mode else None
        for txt in hf_stream_text(HF_DATASETS["s2orc_biomed"], field_candidates=["text","abstract","paper_text"], fraction=frac):
            fo.write(json.dumps({"text": txt}, ensure_ascii=False) + "\n")
            
        # 3) EPFL guidelines (full)
        for txt in hf_stream_text(HF_DATASETS["epfl_guidelines"], field_candidates=["text","guidelines"], fraction=None if not dev_mode else 0.2):
            fo.write(json.dumps({"text": txt}, ensure_ascii=False) + "\n")
            
        # 4) Kaggle CSV if exists
        if Path(KAGGLE_CSV).exists():
            for qa in csv_stream_qas(KAGGLE_CSV):
                fo.write(json.dumps({"text": qa}, ensure_ascii=False) + "\n")
        else:
            print("[WARN] Kaggle CSV not found:", KAGGLE_CSV)
            
    print("[DATA] merged file built:", out_path)
    return str(out_path)

# Create (dev_mode=True uses small fractions; set False for full run)
merged_path = build_merged_jsonl(MERGED_JSONL, dev_mode=True)
print(f" Stage 2: Data merged into {merged_path}")

# Tokenizer

In [ ]:
# Cell: load tokenizer (uses uploaded tokenizer files if present)
tokenizer = None
try:
    # try load from local tokenizer files if present
    if all(Path(TOKENIZER_DIR, TOKENIZER_FILES[k]).exists() for k in ["tokenizer_json","tokenizer_model","tokenizer_config"]):
        print("[TOKENIZER] Loading tokenizer from local files")
        tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, use_fast=True)
    else:
        print("[TOKENIZER] Local tokenizer files not fully present, loading base from HF")
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
except Exception as e:
    print(f"[ERROR] loading tokenizer: {e}, falling back to base model.")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

print("[TOKENIZER] vocab_size:", getattr(tokenizer, "vocab_size", None))
# ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    print("[TOKENIZER] Added PAD token")

print(" Stage 3: Tokenizer loaded.")

# Tokenization

In [ ]:
# Cell: tokenization pipeline
print("[DATA] loading merged jsonl:", merged_path)
raw_ds = load_dataset("json", data_files=merged_path, split="train")
print("[DATA] samples:", len(raw_ds))

max_length = 1024 # Llama 2 context size can be larger (e.g., 2048 or 4096)

def tokenize_fn(examples):
    texts = examples["text"]
    # Ensure PAD token is used for padding
    out = tokenizer(texts, truncation=True, max_length=max_length, padding="max_length")
    out["labels"] = out["input_ids"].copy()
    return out

tokenized = raw_ds.map(
    tokenize_fn, 
    batched=True, 
    num_proc=NUM_PROC, 
    remove_columns=raw_ds.column_names
)
print("[TOKENIZED] done — length:", len(tokenized))
print(" Stage 4: Dataset tokenized.")

# Model & Trainer Setup

In [ ]:
# Cell: load model & prepare Trainer (LoRA or full)
set_seed(TRAINING_ARGS["seed"])

if MODE == "lora":
    # LoRA flow
    print("[MODEL] Loading base model (frozen) for LoRA")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, 
        torch_dtype=torch.float16, 
        device_map="auto", 
        low_cpu_mem_usage=True
    )
    # Resize token embeddings if we added a PAD token
    base_model.resize_token_embeddings(len(tokenizer))
    
    base_model = prepare_model_for_kbit_training(base_model)
    lora_cfg = LoraConfig(
        r=LORA_CONFIG["r"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        target_modules=LORA_CONFIG["target_modules"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias=LORA_CONFIG["bias"],
        task_type=LORA_CONFIG["task_type"]
    )
    model = get_peft_model(base_model, lora_cfg)
    print("[MODEL] LoRA wrappers added. Trainable params:")
    model.print_trainable_parameters()
else:
    # Full fine-tune
    print("[MODEL] Loading base model for full fine-tuning (this is heavy)")
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, 
        torch_dtype=torch.float16, 
        device_map="auto", 
        low_cpu_mem_usage=True
    )
    # Resize token embeddings if we added a PAD token
    model.resize_token_embeddings(len(tokenizer))

# Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# Build TrainingArguments
hf_training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    **TRAINING_ARGS # Pass all args
)

trainer = Trainer(
    model=model,
    args=hf_training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print(f" Stage 5: Trainer Ready. MODE = {MODE}")

# RUN Training

In [ ]:
# Cell: Run Training
print(f"[TRAINER] Starting training... (MODE = {MODE})")

trainer.train()

print("[TRAINER] Training complete.")
print(f" Stage 6: Model training finished. Checkpoints at {CHECKPOINT_DIR}")

# Merge & Save

In [ ]:
# Cell: Save final model (Merge LoRA if used)
print("[SAVE] Saving final model...")

if MODE == "lora":
    print("[SAVE] Merging LoRA adapters into base model for final export...")
   
    try:
       
        merged_model = model.merge_and_unload()
        print("[SAVE] LoRA merge successful.")
        
        # حفظ
        merged_model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"[SAVE] Full MERGED model and tokenizer saved to {OUTPUT_DIR}")

    except Exception as e:
        print(f"[ERROR] Could not merge LoRA adapters: {e}")
        print("[SAVE] Saving LoRA adapter only as fallback...")
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)

else:
    # Full fine-tune
    print("[SAVE] Saving full fine-tuned model...")
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"[SAVE] Full model and tokenizer saved to {OUTPUT_DIR}")

print(f" Stage 7: Final model saved to {OUTPUT_DIR}")

# Sample Generation

In [ ]:
# Cell: sample generation function
def generate_sample(prompt: str, max_new_tokens=200):
    
    active_model = trainer.model.eval()
    
    inputs = tokenizer(prompt, return_tensors="pt").to(next(active_model.parameters()).device)
    
    with torch.no_grad():
        out = active_model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens, 
            do_sample=True, 
            temperature=0.7, 
            top_p=0.95
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Example (run after training)
print("--- Example prompt (after training) ---")
print(generate_sample("Q: What are the common symptoms of anemia?\nA:"))
print("----------------------------------------")
print(f" Stage 8: Sample generation complete.")

# Quantize & Shard

In [ ]:


def export_sharded_state_dict(model, out_dir=QUANT_DIR, n_shards=3):
    os.makedirs(out_dir, exist_ok=True)
   
    state = {k: v.detach().cpu() for k, v in model.named_parameters()}
    items = list(state.items())
    sizes = [v.element_size() * v.nelement() for _, v in items]
    total = sum(sizes)
    target = total / n_shards
    
    shards = [ {} for _ in range(n_shards) ]
    shard_sizes = [0]*n_shards
    cur = 0
    
    for (k, v), sz in zip(items, sizes):
        if shard_sizes[cur] + sz > target and cur < n_shards-1:
            cur += 1
        shards[cur][k] = v
        shard_sizes[cur] += sz
        
    for i, sh in enumerate(shards):
        out_path = os.path.join(out_dir, f"model-0000{i+1}-of-0000{n_shards:05d}.safetensors")
        save_file(sh, out_path)
        print(f"Saved shard: {out_path}, bytes: {shard_sizes[i]}")
        
    # write index file
    weight_map = {}
    for i, sh in enumerate(shards):
        fn = f"model-0000{i+1}-of-0000{n_shards:05d}.safetensors"
        for k in sh.keys():
            weight_map[k] = fn
            
    index = {"metadata": {"total_size": total}, "weight_map": weight_map}
    with open(os.path.join(out_dir, "model.safetensors.index.json"), "w") as f:
        json.dump(index, f, indent=2)
    print(f"[EXPORT] shards and index saved to {out_dir}")

# start
if HAS_GPTQ:
    try:
        print("[GPTQ] Starting quantization (auto-gptq). This step can take long.")
        
        # 1. Quantize
       
        q_model = AutoGPTQForCausalLM.from_pretrained(
            OUTPUT_DIR,
            use_safetensors=True,
            device_map="auto",
            load_in_8bit=False 
        )
        print("[GPTQ] Quantized model loaded into memory.")
        
        q_model.save_quantized(QUANT_DIR)
       
        tokenizer.save_pretrained(QUANT_DIR) 
        print(f"[GPTQ] Saved quantized model config/tokenizer to {QUANT_DIR}")

        
        print("[EXPORT] Exporting quantized model to shards...")
       
        export_sharded_state_dict(q_model, out_dir=QUANT_DIR, n_shards=3)
        
        print(f" Stage 9: Quantization and Sharding complete. Artifacts at {QUANT_DIR}")
        
    except Exception as e:
        print(f"[GPTQ/EXPORT] Error: {e}")
        print("[GPTQ/EXPORT] If you already have quantized shards, skip this step.")
else:
    print("[SKIP] Skipping Stage 9 (Quantization/Sharding) because auto-gptq is not installed.")